## Завдання 1. Setup і typed-схема Olist

In [2]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
from pathlib import Path
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 49.5 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)

def q(sql: str) -> pd.DataFrame:
    """SELECT -> DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql: str) -> None:
    """DDL / DML (можна кілька statement-ів) в одній транзакції.
    Виконується напряму через psycopg2-cursor без параметрів, тому '%' (оператор modulo у stub-seed) не трактується як placeholder."""
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql)
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()

def explain(sql: str) -> str:
    """Виконати EXPLAIN (...) і надрукувати план як текст; план також повертається для подальшого аналізу."""
    with engine.connect() as conn:
        plan = '\n'.join(row[0] for row in conn.execute(text(sql)))
    print(plan)
    return plan

### Отримання даних
Порядок: локальні CSV у `data/olist/` → `kagglehub` → stub-схема. Щоб примусово працювати на stub, задайте `FORCE_STUB = True`
(або змінну середовища `HW5_FORCE_STUB=1`).

In [4]:
FORCE_STUB = os.environ.get('HW5_FORCE_STUB') == '1'
LOCAL_DIR = Path('data/olist')

DATA_MODE, dataset_dir = 'stub', None
if not FORCE_STUB:
    if (LOCAL_DIR / 'olist_orders_dataset.csv').exists():
        DATA_MODE, dataset_dir = 'olist', LOCAL_DIR                      # Варіант B
    else:
        try:
            import kagglehub                                              # Варіант A
            dataset_dir = Path(kagglehub.dataset_download('olistbr/brazilian-ecommerce'))
            DATA_MODE = 'olist'
        except Exception as exc:
            print('Kaggle недоступний, переходимо на stub-схему:', repr(exc)[:200])

print('DATA_MODE =', DATA_MODE, '| dataset_dir =', dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
DATA_MODE = olist | dataset_dir = /kaggle/input/brazilian-ecommerce


In [5]:
files = {
    'olist_customers_raw':   'olist_customers_dataset.csv',
    'olist_orders_raw':      'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw':    'olist_products_dataset.csv',
    'olist_sellers_raw':     'olist_sellers_dataset.csv',
}

if DATA_MODE == 'olist':
    for table_name, file_name in files.items():
        df = pd.read_csv(os.path.join(dataset_dir, file_name))
        df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
        print(f'{table_name:<23} {df.shape}')
else:
    print('stub mode: raw-шар не потрібен')

olist_customers_raw     (99441, 5)
olist_orders_raw        (99441, 8)
olist_order_items_raw   (112650, 7)
olist_products_raw      (32951, 9)
olist_sellers_raw       (3095, 4)


### Typed-схема і `INSERT ... SELECT` (режим `olist`)

In [6]:
TYPED_SQL = '''
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,          -- у CSV саме 'lenght'
    product_description_lenght::INTEGER,   -- у CSV саме 'lenght'
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
'''

if DATA_MODE == 'olist':
    run(TYPED_SQL)
    print('typed-схема створена й заповнена з raw')

typed-схема створена й заповнена з raw


### Stub-schema fallback (режим `stub`)

In [7]:
STUB_SQL = '''
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_state           CHAR(2) NOT NULL,
    customer_city            TEXT
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

INSERT INTO olist_sellers (seller_id, seller_zip_code_prefix, seller_city, seller_state)
VALUES
    ('s_01', 10001, 'Sao Paulo', 'SP'),
    ('s_02', 20001, 'Rio de Janeiro', 'RJ'),
    ('s_03', 30001, 'Belo Horizonte', 'MG');

INSERT INTO olist_products (
    product_id, product_category_name,
    product_name_length, product_description_length,
    product_photos_qty, product_weight_g,
    product_length_cm, product_height_cm, product_width_cm
)
VALUES
    ('p_01', 'electronics', 10, 100, 2,  800, 20, 10, 15),
    ('p_02', 'electronics', 12, 120, 3, 1200, 25, 12, 18),
    ('p_03', 'home',        8,  90, 1, 1500, 30, 20, 25),
    ('p_04', 'home',        9,  80, 2,  700, 18, 10, 14),
    ('p_05', 'fashion',     7,  60, 1,  300, 10,  5,  8),
    ('p_06', 'beauty',      6,  50, 1,  200,  8,  4,  6),
    ('p_07', 'sport',       9,  70, 2,  900, 20, 15, 16);

INSERT INTO olist_customers (
    customer_id, customer_unique_id, customer_zip_code_prefix,
    customer_state, customer_city
)
SELECT
    'c_' || LPAD(n::TEXT, 3, '0'),
    'u_' || LPAD(((n - 1) % 25 + 1)::TEXT, 3, '0'),
    10000 + n,
    (ARRAY['SP', 'RJ', 'MG', 'BA', 'PR'])[(n % 5) + 1],
    'City' || (n % 10)
FROM generate_series(1, 30) AS n;

INSERT INTO olist_orders (
    order_id, customer_id, order_status,
    order_purchase_timestamp, order_approved_at,
    order_delivered_carrier_date, order_delivered_customer_date,
    order_estimated_delivery_date
)
SELECT
    'o_' || LPAD(n::TEXT, 3, '0'),
    'c_' || LPAD(((n % 30) + 1)::TEXT, 3, '0'),
    (ARRAY['delivered', 'delivered', 'delivered', 'canceled', 'shipped'])[(n % 5) + 1],
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days'),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '1 day',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '3 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '8 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '10 days'
FROM generate_series(1, 50) AS n;

INSERT INTO olist_order_items (
    order_id, order_item_id, product_id, seller_id,
    shipping_limit_date, price, freight_value
)
SELECT
    'o_' || LPAD(((n - 1) % 50 + 1)::TEXT, 3, '0'),
    ((n - 1) / 50) + 1,
    'p_0' || ((n % 7) + 1),
    's_0' || ((n % 3) + 1),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '5 days',
    (n % 200 + 10)::NUMERIC(12, 2),
    (n % 30 + 5)::NUMERIC(12, 2)
FROM generate_series(1, 100) AS n;
'''

if DATA_MODE == 'stub':
    run(STUB_SQL)
    print('stub-схема створена: 30 customers, 50 orders, 100 order_items')

### Smoke-test кількості рядків

In [8]:
q('''
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
''')

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


In [9]:
if DATA_MODE == 'olist':
    display(q('''
    SELECT 'customers' AS table_name, (SELECT COUNT(*) FROM olist_customers_raw) AS n_raw,
           (SELECT COUNT(*) FROM olist_customers) AS n_typed
    UNION ALL SELECT 'orders',      (SELECT COUNT(*) FROM olist_orders_raw),      (SELECT COUNT(*) FROM olist_orders)
    UNION ALL SELECT 'order_items', (SELECT COUNT(*) FROM olist_order_items_raw), (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL SELECT 'products',    (SELECT COUNT(*) FROM olist_products_raw),    (SELECT COUNT(*) FROM olist_products)
    UNION ALL SELECT 'sellers',     (SELECT COUNT(*) FROM olist_sellers_raw),     (SELECT COUNT(*) FROM olist_sellers)
    ORDER BY table_name
    '''))

,table_name,n_raw,n_typed
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,sellers,3095,3095


**Висновок.** `INSERT ... SELECT` не має `WHERE`, тож typed-шар не може «тихо» втратити рядки: будь-яке порушення PK/FK/`NOT NULL`/`CHECK` зупинило б увесь `INSERT` з помилкою.
Для реального Olist кількості raw і typed збігаються (99 441 customers і orders, 112 650 order_items, 32 951 products, 3 095 sellers).
У stub-режимі порівняння з raw не виконується, а очікувані обсяги — 30 / 50 / 100 / 7 / 3.

> ✏️ Звірте числа з фактичним виводом.

## Завдання 2. CTE-chain pipeline (8 stages)

Кожен stage виконує одну трансформацію:

| Stage | CTE | Грануляність | Що робить |
|---|---|---|---|
| 0 | `reference_date` | 1 рядок | дата «зрізу» = `MAX(order_purchase_timestamp)::DATE + 1` |
| 1 | `stage1_orders_total` | замовлення | сума замовлення (price + freight) і статус |
| 2 | `stage2_rfm` | customer | `recency_days`, `frequency`, `monetary`, `first_order_date` |
| 3 | `stage3_cohort` | customer | `cohort_quarter = DATE_TRUNC('quarter', first_order_date)` |
| 4 | `stage4_rolling_aov` | замовлення | `AVG(order_total)` у вікні `RANGE 30 days PRECEDING` + `rn_recent` |
| 5 | `stage5_cancel_rate` | customer | частка `canceled` |
| 6 | `stage6_category_revenue` | customer × category | revenue за категорією |
| 7 | `stage7_preferred_category` | customer | top-1 категорія через `ROW_NUMBER()` |
| 8 | фінальний `SELECT` | customer | `LEFT JOIN` усіх stage до `olist_customers`, `COALESCE` для відсутніх значень |

SQL зберігається в Python-змінних `PIPELINE_CTES` і `FINAL_SELECT`, щоб **той самий** текст використати в MV (завдання 4) і в `EXPLAIN` (завдання 5).

In [10]:
PIPELINE_CTES = '''
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status (грануляність: одне замовлення)
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
'''

FINAL_SELECT = '''
-- Stage 8: final customer-level feature row (один рядок на customer_id)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
'''

PIPELINE_SQL = PIPELINE_CTES + '\n' + FINAL_SELECT

features = q(PIPELINE_SQL + '\nORDER BY monetary DESC NULLS LAST, c.customer_id\nLIMIT 20;')
features

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


Окрема перевірка кожного stage: кількість рядків і ключова статистика (single responsibility → кожен етап можна валідувати ізольовано).

In [11]:
q(PIPELINE_CTES + '''
SELECT 'stage1_orders_total' AS stage, COUNT(*) AS n_rows,
       COUNT(DISTINCT customer_id) AS n_customers, SUM(order_total)::NUMERIC AS check_value, 'SUM(order_total)' AS check_name
FROM stage1_orders_total
UNION ALL
SELECT 'stage2_rfm', COUNT(*), COUNT(DISTINCT customer_id), MAX(frequency), 'MAX(frequency)'
FROM stage2_rfm
UNION ALL
SELECT 'stage3_cohort', COUNT(*), COUNT(DISTINCT customer_id), COUNT(DISTINCT cohort_quarter), 'n cohorts'
FROM stage3_cohort
UNION ALL
SELECT 'stage4_rolling_aov', COUNT(*), COUNT(DISTINCT customer_id), COUNT(*) FILTER (WHERE rn_recent = 1), 'rows with rn_recent = 1'
FROM stage4_rolling_aov
UNION ALL
SELECT 'stage5_cancel_rate', COUNT(*), COUNT(DISTINCT customer_id), ROUND(AVG(cancellation_rate), 4), 'AVG(cancellation_rate)'
FROM stage5_cancel_rate
UNION ALL
SELECT 'stage6_category_revenue', COUNT(*), COUNT(DISTINCT customer_id), COUNT(DISTINCT product_category_name), 'n categories'
FROM stage6_category_revenue
UNION ALL
SELECT 'stage7_preferred_category', COUNT(*), COUNT(DISTINCT customer_id), NULL, 'must be 1 row per customer'
FROM stage7_preferred_category;
''')

,stage,n_rows,n_customers,check_value,check_name
0,stage1_orders_total,99441,99441,1.584355e+07,SUM(order_total)
1,stage2_rfm,99441,99441,1.000000e+00,MAX(frequency)
2,stage3_cohort,99441,99441,1.000000e+01,n cohorts
3,stage4_rolling_aov,99441,99441,9.944100e+04,rows with rn_recent = 1
4,stage5_cancel_rate,99441,99441,6.300000e-03,AVG(cancellation_rate)
5,stage6_category_revenue,99470,98666,7.400000e+01,n categories
6,stage7_preferred_category,98666,98666,NaN,must be 1 row per customer


In [12]:
# Розподіл frequency і частка покупців з rolling AOV, що відрізняється від суми останнього замовлення
q(PIPELINE_CTES + '''
SELECT
    frequency,
    COUNT(*) AS n_customers,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM stage2_rfm
GROUP BY frequency
ORDER BY frequency;
''')

,frequency,n_customers,avg_monetary,avg_recency_days
0,1,99441,159.33,291.0


**Interpretation.**
- У stage4–stage7 кількість рядків з `rn_recent = 1` і рядків stage7 дорівнює кількості покупців у stage2 — top-1 ранжування не дублює і не губить клієнтів.
- **Важливе обмеження Olist:** pipeline рахується за `customer_id`, а в Olist це ідентифікатор «покупець у конкретному замовленні». На реальних даних кожен `customer_id` має рівно одне замовлення,
  тому `frequency = 1` для всіх, `cancellation_rate` — лише 0 або 1, а `avg_order_value_30d_recent` дорівнює сумі цього єдиного замовлення. Фічі формально коректні, але вироджені.
  Для справжнього RFM pipeline треба групувати за `customer_unique_id` (як у HW4) — це зміна однієї колонки в stage1/stage2.
  На stub-схемі `customer_id` має 1–2 замовлення, тому там rolling-вікно й частка скасувань уже показують різні значення.
- `recency_days` рахується від `reference_date`, а не від `CURRENT_DATE`, тож результат не «старіє» щодня.
- Покупці без замовлень отримують `recency_days = 9999`, `frequency = 0` і `preferred_category = '(none)'` — sentinel-значення, які в ML краще замінити на прапорець `has_orders`.

> ✏️ Звірте висновки з фактичним виводом (топ-покупці, розподіл frequency, кількість когорт).

## Завдання 3. Recursive CTE для referral chain

In [13]:
run('''
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
''')

In [14]:
q('''
SELECT referrer_customer_id, referred_customer_id, referred_at
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
''')

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-11 13:37:16.764574+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-11 13:37:16.764574+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-11 13:37:16.764574+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-11 13:37:16.764574+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-11 13:37:16.764574+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-11 13:37:16.764574+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-11 13:37:16.764574+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-11 13:37:16.764574+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-11 13:37:16.764574+00:00


In [15]:
q('''
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)   -- cycle guard
      AND rd.depth < 50                                   -- depth guard
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
''')

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


**Interpretation.** Синтетична таблиця — це один ланцюжок з 10 покупців (перші 10 `customer_id` за алфавітом), тому на кожній глибині 2–10 рівно один покупець,
а `sample_path` на глибині 10 показує повний шлях. Глибина 1 — усі покупці, яких ніхто не запросив: це решта клієнтів плюс корінь ланцюжка.
Anchor використовує `NOT EXISTS`, а не `NOT IN`: якщо в підзапиті `NOT IN` з'явиться хоча б один `NULL`, умова для всіх рядків стане `NULL` і anchor поверне 0 рядків.

### Демонстрація циклу A → B → A
Щоб не ламати основну таблицю, копіюємо її в `hw5_referrals_cycle_demo` і замикаємо ланцюжок: останній покупець «запрошує» першого.
Тепер граф — кільце з 10 вершин, і **кореня немає** — anchor з `NOT EXISTS` не знаходить жодного учасника кільця.
Тому для демонстрації запускаємо обхід від конкретного покупця.

In [16]:
run('''
DROP TABLE IF EXISTS hw5_referrals_cycle_demo;

CREATE TABLE hw5_referrals_cycle_demo AS
SELECT referrer_customer_id, referred_customer_id FROM hw5_referrals;

INSERT INTO hw5_referrals_cycle_demo (referrer_customer_id, referred_customer_id)
SELECT
    (SELECT MAX(referred_customer_id) FROM hw5_referrals),     -- останній у ланцюжку
    (SELECT MIN(referrer_customer_id) FROM hw5_referrals);     -- перший у ланцюжку
''')

In [17]:
CYCLE_ANCHOR = '''
    SELECT
        MIN(referrer_customer_id) AS customer_id,
        1 AS depth,
        ARRAY[MIN(referrer_customer_id)]::TEXT[] AS path_ids
    FROM hw5_referrals
'''

def cycle_query(guard_sql: str) -> str:
    return f'''
WITH RECURSIVE walk AS (
    {CYCLE_ANCHOR}
    UNION ALL
    SELECT r.referred_customer_id, w.depth + 1, w.path_ids || r.referred_customer_id
    FROM hw5_referrals_cycle_demo AS r
    JOIN walk AS w ON w.customer_id = r.referrer_customer_id
    WHERE {guard_sql}
)
SELECT COUNT(*) AS n_rows, MAX(depth) AS max_depth, COUNT(DISTINCT customer_id) AS n_distinct_customers
FROM walk;
'''

pd.concat([
    q(cycle_query('NOT r.referred_customer_id = ANY(w.path_ids) AND w.depth < 50')).assign(guards='path_ids + depth < 50'),
    q(cycle_query('w.depth < 50')).assign(guards='only depth < 50'),
], ignore_index=True)

,n_rows,max_depth,n_distinct_customers,guards
0,10,10,10,path_ids + depth < 50
1,50,50,10,only depth < 50


Той самий захист через стандартний `CYCLE`-clause (PostgreSQL 14+): PostgreSQL сам веде шлях і позначає рядок, на якому цикл замкнувся.

In [18]:
q('''
WITH RECURSIVE walk (customer_id, depth) AS (
    SELECT MIN(referrer_customer_id), 1
    FROM hw5_referrals
    UNION ALL
    SELECT r.referred_customer_id, w.depth + 1
    FROM hw5_referrals_cycle_demo AS r
    JOIN walk AS w ON w.customer_id = r.referrer_customer_id
)
CYCLE customer_id SET is_cycle USING cycle_path
SELECT depth, customer_id, is_cycle
FROM walk
ORDER BY depth;
''')

,depth,customer_id,is_cycle
0,1,00012a2ce6f8dcda20d059ce98491703,False
1,2,000161a058600d5901f007fab4c27140,False
2,3,0001fd6190edaaf884bcaf3d49edf079,False
3,4,0002414f95344307404f0ace7a26f1d5,False
4,5,000379cdec625522490c315e70c7a9fb,False
5,6,0004164d20a9e969af783496f3408652,False
6,7,000419c5494106c306a97b5635748086,False
7,8,00046a560d407e99b969756e0b10f282,False
8,9,00050bf6e01e69d5c0fd612f1bcfb69c,False
9,10,000598caf2ef4117407665ac33275130,False


**Висновок.** З обома захистами обхід зупиняється, щойно наступний покупець уже є в `path_ids`: 10 рядків, 10 різних покупців.
Лише з depth guard рекурсія ходить по колу до `depth = 50` — 50 рядків, але лише 10 різних покупців, тобто 40 рядків дублікатів, які зіпсували б агрегати.
Без жодного захисту запит не завершився б ніколи (`UNION ALL` не прибирає повтори). `CYCLE`-clause дає той самий результат декларативно: рядок з `is_cycle = true` — це точка замикання, і далі рекурсія не йде.

## Завдання 4. Materialized View з `REFRESH ... CONCURRENTLY`
MV будується з **того самого** `PIPELINE_SQL`, що й у завданні 2 (усі 8 stages, усі features).

In [19]:
run(f'''
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
{PIPELINE_SQL};

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

ANALYZE mv_olist_customer_features;
''')

q('''
SELECT pg_get_viewdef('mv_olist_customer_features'::regclass) LIKE '%stage7_preferred_category%' AS mv_contains_full_pipeline,
       (SELECT COUNT(*) FROM mv_olist_customer_features) AS n_rows,
       (SELECT COUNT(*) FROM olist_customers)            AS n_customers
''')

,mv_contains_full_pipeline,n_rows,n_customers
0,True,99441,99441


In [20]:
q('''
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
''')

,n_rows
0,99441


In [21]:
q('''
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
''')

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


Concurrent refresh — окремою клітинкою після `UNIQUE INDEX`:

In [22]:
import time
t0 = time.perf_counter()
run('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;')
print(f'REFRESH CONCURRENTLY: {time.perf_counter() - t0:.2f} s')

REFRESH CONCURRENTLY: 4.76 s


In [23]:
# Для порівняння: звичайний REFRESH (бере ACCESS EXCLUSIVE lock — читання MV блокуються на весь час перерахунку)
t0 = time.perf_counter()
run('REFRESH MATERIALIZED VIEW mv_olist_customer_features;')
print(f'REFRESH (blocking): {time.perf_counter() - t0:.2f} s')

q('''
SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname
''')

REFRESH (blocking): 4.73 s


,indexname,indexdef
0,idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


**Пояснення.** `REFRESH ... CONCURRENTLY` будує нову версію результату у тимчасовій таблиці, порівнює її з поточною за унікальним ключем і застосовує лише різницю
(`INSERT`/`UPDATE`/`DELETE`). Тому під час refresh MV залишається доступною для читання. Ціна — повільніше за звичайний `REFRESH`, і обов'язковий `UNIQUE INDEX`
на звичайних колонках (не partial, не expression) — саме за ним PostgreSQL зіставляє старі й нові рядки. `customer_id` у MV унікальний, бо фінальний `SELECT` іде від `olist_customers` (PK) і всі stage-и мають по одному рядку на покупця.

## Завдання 5. EXPLAIN ANALYZE
### 5.1. Частковий CTE-pipeline (stages 0–2), як у шаблоні

In [24]:
plan_partial = explain('''
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
''')

Limit  (cost=33611.15..33611.17 rows=10 width=65) (actual time=642.885..642.894 rows=10 loops=1)
  Buffers: shared hit=105261, temp read=961 written=1696
  ->  Sort  (cost=33611.15..33611.65 rows=200 width=65) (actual time=642.882..642.889 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=105261, temp read=961 written=1696
        ->  Subquery Scan on stage2_rfm  (cost=33601.33..33606.83 rows=200 width=65) (actual time=466.017..608.881 rows=99441 loops=1)
              Buffers: shared hit=105261, temp read=961 written=1696
              ->  HashAggregate  (cost=33601.33..33604.83 rows=200 width=69) (actual time=466.015..595.755 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=105261, temp read=961 writ

### 5.2. Повний CTE-pipeline (усі 8 stages) — щоб побачити найдорожчий feature

In [25]:
plan_full = explain('EXPLAIN (ANALYZE, BUFFERS)\n' + PIPELINE_SQL + '\nORDER BY monetary DESC NULLS LAST\nLIMIT 10;')

Limit  (cost=115110.55..115110.57 rows=10 width=150) (actual time=4382.459..4384.680 rows=10 loops=1)
  Buffers: shared hit=111711, temp read=8438 written=8587
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27069.86 rows=99441 width=103) (actual time=0.077..608.924 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24484.77 rows=112650 width=95) (actual time=0.046..360.339 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12563.75 rows=99441 width=83) (actual time=0.023..132.622 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.013..49.982 rows=112650 loops=1)
          

### 5.3. Lookup з MV за `customer_id`

In [26]:
plan_lookup = explain('''
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
''')

Nested Loop  (cost=0.83..8.91 rows=1 width=92) (actual time=0.044..0.046 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.47 rows=1 width=33) (actual time=0.020..0.021 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..5328.03 rows=99441 width=33) (actual time=0.019..0.020 rows=1 loops=1)
              Heap Fetches: 0
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=92) (actual time=0.008..0.008 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=64 read=5
Planning Time: 0.429 ms
Execution Time: 0.069 ms


### 5.4. Top-N з MV
Тонкість: шаблонний індекс `(monetary DESC)` має порядок `DESC NULLS FIRST` (так працює `DESC` за замовчуванням), а запит сортує `DESC NULLS LAST`.
Порядки не збігаються, тому такий індекс **не** підходить для `ORDER BY ... LIMIT`. Порівнюємо обидва варіанти.

In [27]:
run('''
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
ANALYZE mv_olist_customer_features;
''')

TOPN = '''
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
'''
plan_topn_template_index = explain(TOPN)

Limit  (cost=4449.01..4450.16 rows=10 width=92) (actual time=35.427..35.603 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=92) (actual time=35.425..35.598 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=92) (actual time=30.588..30.591 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=92) (actual time=0.010..8.599 rows=49720 loops=2)
                    Buffers: shared hit=1600
Planning:
  Buffers: shared hit=64 read=1
Planning Time: 0.331 ms
Execution Time: 35.634 ms


In [28]:
run('''
DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;
CREATE INDEX idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC NULLS LAST);
ANALYZE mv_olist_customer_features;
''')
plan_topn_matching_index = explain(TOPN)

Limit  (cost=0.42..1.32 rows=10 width=92) (actual time=0.025..0.039 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features  (cost=0.42..8995.39 rows=99441 width=92) (actual time=0.023..0.036 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=63 read=1
Planning Time: 0.260 ms
Execution Time: 0.056 ms


Зведення часу виконання з усіх планів:

In [29]:
import re

def exec_ms(plan: str) -> float:
    return float(re.search(r'Execution Time: ([\d.]+) ms', plan).group(1))

def top_nodes(plan: str, k: int = 6) -> str:
    nodes = re.findall(r'->\s+([A-Za-z][A-Za-z ]+?)(?:\s+on|\s+using|\s+\(|$)', plan, flags=re.M)
    first = plan.splitlines()[0].split('  (')[0].strip()
    return ', '.join(dict.fromkeys([first] + [n.strip() for n in nodes]))  # унікальні, у порядку появи

pd.DataFrame([
    ('5.1 pipeline stages 0–2',           exec_ms(plan_partial),              top_nodes(plan_partial)),
    ('5.2 full pipeline (8 stages)',      exec_ms(plan_full),                 top_nodes(plan_full)),
    ('5.3 MV lookup by customer_id',      exec_ms(plan_lookup),               top_nodes(plan_lookup)),
    ('5.4a MV top-N, index DESC',         exec_ms(plan_topn_template_index),  top_nodes(plan_topn_template_index)),
    ('5.4b MV top-N, index DESC NULLS LAST', exec_ms(plan_topn_matching_index), top_nodes(plan_topn_matching_index)),
], columns=['scenario', 'execution_ms', 'plan_nodes'])

,scenario,execution_ms,plan_nodes
0,5.1 pipeline stages 0–2,644.490,"Limit, Sort, Subquery Scan, HashAggregate, Nes..."
1,5.2 full pipeline (8 stages),4393.281,"Limit, GroupAggregate, Merge Left Join, Index ..."
2,5.3 MV lookup by customer_id,0.069,"Nested Loop, Limit, Index Only Scan, Index Scan"
3,"5.4a MV top-N, index DESC",35.634,"Limit, Gather Merge, Sort, Parallel Seq Scan"
4,"5.4b MV top-N, index DESC NULLS LAST",0.056,"Limit, Index Scan"


**Interpretation планів** (опис для реального Olist; на stub-схемі з 30–100 рядками плани ті самі за формою, але всі часи в межах мілісекунди, і планувальник може обирати `Seq Scan` навіть за наявності індексу — читати 1 сторінку дешевше, ніж індекс):
- **5.1 / 5.2:** `Seq Scan` по `olist_orders` (≈ 99 тис.) і `olist_order_items` (≈ 113 тис.) → `Hash Right Join` → `HashAggregate` (stage1, групування за 4 колонками) → ще один `HashAggregate` (stage2).
  У повному pipeline додаються `Sort` + `WindowAgg` (stage4: два вікна по `customer_id`, одне з `RANGE 30 days`), другий прохід join-у з `olist_products` і `Sort` + `WindowAgg` для top-1 категорії (stage7), а також `CTE Scan` для `stage1_orders_total`, бо на нього посилаються кілька stage-ів і PostgreSQL його матеріалізує.
- **5.3:** `Index Scan using idx_mv_olist_customer_features_customer_id` — одна точкова вибірка з MV; час — частки мілісекунди проти сотень мілісекунд повного pipeline.
- **5.4:** з індексом `(monetary DESC)` план — `Seq Scan` + `Sort (top-N heapsort)` по всьому MV; з індексом `(monetary DESC NULLS LAST)` — `Limit` → `Index Scan`, який читає лише 10 рядків.

> ✏️ Звірте назви вузлів і час з фактичними планами та таблицею вище; підставте конкретні мілісекунди в Reflection.

## Reflection

**Найдорожчий feature.** За планом 5.2 найбільше часу йде на stage1 (`Hash Join` orders × order_items + `HashAggregate` по ~100 тис. замовлень), бо від нього залежать RFM, rolling AOV і cancellation rate.
Серед власне features найдорожчі `avg_order_value_30d` (stage4) і `preferred_category` (stage6–7): обидва вимагають `Sort` перед `WindowAgg`, а stage6 ще раз сканує й join-ить order_items з products.
Сам `reference_date` дешевий — один `Aggregate` над `Seq Scan`.

**Кешувати через MV?** Так. Різниця між повним pipeline (5.2) і lookup з MV (5.3) — кілька порядків: `Index Scan` по унікальному індексу замість двох `HashAggregate`, двох `WindowAgg` і кількох `Sort`.
Для Olist (історичний датасет) достатньо refresh **on-demand** після завантаження нових даних; у живому e-commerce — **щодоби** вночі: RFM і когорти не змінюються щохвилини, а `REFRESH CONCURRENTLY` не блокує читання моделі під час перерахунку.

**Коли краще звичайний VIEW.** Коли потрібна абсолютна свіжість (статус замовлення для служби підтримки, антифрод у момент оплати) або запит дешевий і рідкісний — тоді MV додає storage і ризик застарілих даних без виграшу.

**Termination recursive CTE.** Рекурсія зупиняється, коли recursive part повертає 0 нових рядків. При циклі A → B → A цього не стається: без захисту `UNION ALL` додає ті самі вершини нескінченно.
Захисти: `NOT next_id = ANY(path_ids)` відкидає вершину, яка вже є на шляху, а `depth < 50` — страховка від дуже довгих або непередбачених графів. У PostgreSQL 14+ те саме робить `CYCLE ... SET is_cycle USING path`.
Наша демонстрація: лише з depth guard — 50 рядків замість 10.

**Головний trade-off** — freshness vs recompute-cost: повний перерахунок коштує сотні мілісекунд і росте з даними, а features для моделей не потребують секундної актуальності. Тому MV з добовим refresh вигідніша; storage-cost (один рядок на покупця) незначний.